# 🔁 Hands-on: build a Claude agent loop, by actually running one

**Colab edition** of Domain 1 (*Agentic Architecture* — the 27% domain) from the
*Claude Certified Architect* prep track. You'll build a real tool-using agent loop
against the Claude API and *watch* the exam's core ideas happen: the model decides,
**your** code acts, and the loop's stop condition is a model decision you must bound.

Each exercise has a **TODO** for you to complete and a **self-check** that prints
✅ / ❌ — the checks never raise, so you can run the whole notebook top-to-bottom
even with exercises unsolved. Cost is a few US cents on `claude-haiku-4-5`.


In [ ]:
# Setup — run me first
%pip install -q anthropic

import os, getpass, json
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key (input is hidden): ")

import anthropic
client = anthropic.Anthropic()

# A cheap, fast model is plenty for learning the loop mechanics. Swap to
# "claude-opus-4-8" when you want the strongest reasoning.
MODEL = "claude-haiku-4-5"

def check(name, fn, hint=""):
    """Run a check; print PASS/FAIL, never raise (so the notebook runs unsolved)."""
    try:
        ok = fn()
    except Exception as e:
        ok = False
        hint = (hint + f"  (raised {type(e).__name__}: {e})").strip()
    print(f"{'✅' if ok else '❌'}  {name}" + (f"  — {hint}" if not ok and hint else ""))
    return ok

print("Client ready ✓   model =", MODEL)


## Exercise 1 — A plain call cannot *act*
**Goal:** ask the model to do arithmetic it's bad at, with no tools. Observe that it
answers from its head — sometimes wrong — because it has no way to *run* anything.

**Learning:** an LLM on its own is a text predictor. Tools are what let it act. This is
the gap the agent loop closes.


In [ ]:
msg = client.messages.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "What is 4839 * 1176? Answer with just the number."}],
)
answer = next(b.text for b in msg.content if b.type == "text").strip()
print("model said:", answer)
print("actually:  ", 4839 * 1176)
print("stop_reason:", msg.stop_reason)


## Exercise 2 — Define a tool
A tool is a **name + description + JSON-Schema for its inputs**. The description is how
the model decides *when* to call it, so be prescriptive.

**TODO:** finish `calc_tool` — give it an `input_schema` with a single required string
property `expression`, and `additionalProperties: false`.


In [ ]:
import ast, operator as op

# a safe arithmetic evaluator — THIS is the code your harness runs (not the model)
_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
        ast.Pow: op.pow, ast.USub: op.neg, ast.Mod: op.mod}
def _ev(node):
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp):   return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("unsupported")
def calculator(expression: str) -> str:
    return str(_ev(ast.parse(expression, mode="eval").body))

# TODO: complete the tool definition
calc_tool = {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression. Call this whenever the user asks for a calculation.",
    # "input_schema": { ... }   # <-- add this
}


In [ ]:
check("calc_tool has a name", lambda: calc_tool.get("name") == "calculator")
check("input_schema is an object schema",
      lambda: calc_tool["input_schema"]["type"] == "object",
      "add input_schema with type: 'object'")
check("requires an 'expression' string",
      lambda: "expression" in calc_tool["input_schema"]["required"]
              and calc_tool["input_schema"]["properties"]["expression"]["type"] == "string",
      "expression must be a required string property")


## Exercise 3 — The manual agent loop
Now the heart of it. Send the tool; if the model returns `stop_reason == "tool_use"`,
**your code** runs the tool and feeds the result back as a `tool_result`. Repeat until
the model stops calling tools.

**TODO:** inside the loop, run the tool and append a `user` message whose content is a
`tool_result` block with the matching `tool_use_id`.


In [ ]:
def run_agent(question, tools, tool_fns, max_steps=6):
    messages = [{"role": "user", "content": question}]
    steps = 0
    while True:
        steps += 1
        resp = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        if resp.stop_reason != "tool_use":
            final = next((b.text for b in resp.content if b.type == "text"), "")
            return {"answer": final, "steps": steps, "messages": messages}

        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = tool_fns[block.name](**block.input)
                # TODO: append a tool_result block for THIS call to tool_results.
                # It needs: type='tool_result', tool_use_id=block.id, content=out
                pass
        messages.append({"role": "user", "content": tool_results})

r = None
try:
    r = run_agent("What is 4839 * 1176?", [calc_tool], {"calculator": calculator})
    print("answer:", r["answer"])
    print("steps: ", r["steps"])
except Exception as e:
    print("loop errored (expected until you finish the TODO):", type(e).__name__, e)


In [ ]:
check("loop returned an answer", lambda: r is not None and bool(r["answer"]))
check("answer contains the correct product",
      lambda: str(4839 * 1176) in r["answer"].replace(",", ""),
      "the model should quote the tool's result, not its own guess")
check("it actually took a tool step (>1 model call)",
      lambda: r["steps"] >= 2, "a tool round-trip is at least 2 model calls")


## Exercise 4 — The stop condition is a model decision — bound it
The exam's favourite trap: *"the model decides when to stop."* A confused model can loop
forever, burning tokens. The fix is an **external cap your code enforces** — not a
system-prompt plea.

**TODO:** add a `max_steps` guard to `run_agent_capped` so it stops after `max_steps`
model calls even if the model is still asking for tools.


In [ ]:
def run_agent_capped(question, tools, tool_fns, max_steps=6):
    messages = [{"role": "user", "content": question}]
    steps = 0
    while True:
        steps += 1
        # TODO: if steps > max_steps: return a dict with hit_cap=True and the steps count
        resp = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        if resp.stop_reason != "tool_use":
            final = next((b.text for b in resp.content if b.type == "text"), "")
            return {"answer": final, "steps": steps, "hit_cap": False}
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                out = tool_fns[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})
        messages.append({"role": "user", "content": results})

# A deliberately annoying tool that never satisfies the model, to test the cap.
def always_more(_: str) -> str:
    return "Not found. Try calling search again with a different query."
loop_tool = {"name": "search", "description": "Search a knowledge base.",
             "input_schema": {"type": "object",
                              "properties": {"query": {"type": "string"}},
                              "required": ["query"], "additionalProperties": False}}
capped = None
try:
    capped = run_agent_capped("Find the airspeed velocity of an unladen swallow in the KB.",
                              [loop_tool], {"search": always_more}, max_steps=4)
    print(capped)
except Exception as e:
    print("errored (expected until TODO done):", type(e).__name__, e)


In [ ]:
check("the cap stopped the loop", lambda: capped is not None and capped.get("hit_cap") is True,
      "without the guard this would loop until the model gives up or forever")
check("it stopped at the cap, not past it", lambda: capped["steps"] <= 5,
      "max_steps=4 means at most ~4-5 model calls")


## Exercise 5 — The SDK tool runner does the loop for you
You just hand-wrote the loop to understand it. In production the SDK's **tool runner**
drives request → execute → feed-back → repeat, so you only write the tool function.

**TODO:** decorate `add` with `@anthropic.beta_tool`, then pass it to
`client.beta.messages.tool_runner(...)`.


In [ ]:
# The @beta_tool decorator turns a typed function + docstring into a tool schema.
@anthropic.beta_tool
def add(a: int, b: int) -> str:
    """Add two integers.

    Args:
        a: first addend.
        b: second addend.
    """
    return str(a + b)

runner_final = None
try:
    runner = client.beta.messages.tool_runner(
        model=MODEL, max_tokens=512, tools=[add],
        messages=[{"role": "user", "content": "What is 25 + 17? Use the tool."}],
    )
    for message in runner:
        runner_final = message
    print(next((b.text for b in runner_final.content if b.type == "text"), ""))
except Exception as e:
    print("errored:", type(e).__name__, e)


In [ ]:
check("tool runner produced a final message", lambda: runner_final is not None)
check("answer contains 42",
      lambda: "42" in next((b.text for b in runner_final.content if b.type == "text"), ""),
      "the runner should have executed add(25, 17)")


## 🌶️ Stretch — two tools, the model chooses
Give the agent both `calculator` and a `search` tool over a tiny fact table, then ask a
question that needs one, and another that needs the other. Watch it route.

**TODO:** implement `search_facts` and add its tool schema to the `tools` list, then run
`run_agent` (from Exercise 3) on both questions.


In [ ]:
FACTS = {"capital of france": "Paris", "speed of light": "299,792,458 m/s"}
def search_facts(query: str) -> str:
    q = query.lower().strip()
    for k, v in FACTS.items():
        if k in q or q in k:
            return v
    return "No matching fact."

# TODO: build search_tool (name 'lookup', required string 'query') and pass BOTH tools.
search_tool = None  # <-- replace

try:
    tools = [calc_tool, search_tool]
    fns = {"calculator": calculator, "lookup": search_facts}
    for q in ["What is 19 * 23?", "What is the capital of France?"]:
        out = run_agent(q, tools, fns)
        print(f"Q: {q}\nA: {out['answer']}\n")
except Exception as e:
    print("errored (expected until TODO done):", type(e).__name__, e)


## 🎯 Self-score
Run this after you've worked through the exercises to see how many checks pass.


In [ ]:
import io, contextlib
checks = []
def _c(name, fn):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        ok = check(name, fn)
    checks.append(ok); print(buf.getvalue(), end="")

_c("E2 tool schema", lambda: calc_tool["input_schema"]["required"] == ["expression"])
_c("E3 manual loop correct", lambda: r and str(4839*1176) in r["answer"].replace(",",""))
_c("E4 cap enforced", lambda: capped and capped.get("hit_cap") is True)
_c("E5 tool runner", lambda: runner_final and "42" in next((b.text for b in runner_final.content if b.type=="text"), ""))
_c("Stretch two tools", lambda: search_tool is not None and search_tool.get("name") == "lookup")
print(f"\nScore: {sum(checks)} / {len(checks)}" + ("  — mastered Domain 1 mechanics 🏆" if all(checks) else ""))


---
**What you just proved (and what the exam tests):**

- An LLM **orchestrates**; it cannot act. Your harness parses the `tool_use`, runs the
  real tool, and returns a `tool_result`. That separation is also the security boundary.
- The loop's **stop condition is a model decision** — so an *external* `max_steps` (and a
  token budget) is the guardrail the model can't override. A system-prompt "please stop"
  is not.
- The SDK **tool runner** is the production shortcut, but it's the same loop underneath.

Next: Domain 2 (Tool Design & MCP) builds on exactly this — designing the tool surface
the model routes over.
